# Staged Transaction Linking Experiment

**Goal**: Two-stage receipt-to-bank-statement matching that processes each image independently.

Unlike the single-pass approach (which sends receipt + bank statement together),
this experiment splits inference into two stages:

- **Stage 1** (receipt image only): Extract store name, date, and total for each receipt
- **Stage 2** (bank statement image only): Dynamic prompt with Stage 1 results as text,
  find matching debit transactions

**Benefits**:
- Each call uses fewer tiles (more resolution per image for real scans)
- Stage 1 output is inspectable — errors caught before Stage 2
- Stage 2 doesn't need to "see" receipts at all — just text from Stage 1
- Mirrors the proven multi-turn bank extraction pattern

In [ ]:
"""Cell 1: Imports and sys.path setup."""

import re
import sys
from pathlib import Path

import torch
import yaml
from IPython.display import display
from PIL import Image

# Add project root to path so we can import from models/, common/, etc.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.pipeline_config import PipelineConfig  # noqa: E402
from common.simple_prompt_loader import SimplePromptLoader  # noqa: E402
from models.internvl3_image_preprocessor import InternVL3ImagePreprocessor  # noqa: E402
from models.registry import get_model  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
"""Cell 2: Load experiment config for staged_multi_receipt variant."""

config_path = PROJECT_ROOT / "config" / "experiment_config.yml"
with config_path.open() as f:
    exp_config = yaml.safe_load(f)

# Base settings
model_type = exp_config["model"]["type"]
max_tiles = exp_config["model"]["max_tiles"]
max_new_tokens = exp_config["model"]["max_new_tokens"]

# Staged multi-receipt overrides
staged = exp_config["staged_multi_receipt"]
max_tiles = staged["model"].get("max_tiles", max_tiles)
max_new_tokens = staged["model"].get("max_new_tokens", max_new_tokens)
prompt_file = staged["prompts"]["file"]
stage1_key = staged["prompts"]["stage1_key"]
stage2_key = staged["prompts"]["stage2_key"]
ground_truth = staged["ground_truth"]

# Stage-specific token limits from prompt YAML settings
prompt_settings = SimplePromptLoader.get_settings(prompt_file)
stage1_max_tokens = prompt_settings.get("max_new_tokens_stage1", 500)
stage2_max_tokens = prompt_settings.get("max_new_tokens_stage2", 1500)

# Data config
data_mode = exp_config["data"]["mode"]
data_dir = PROJECT_ROOT / exp_config["data"]["dir"]

print(f"Model: {model_type}, max_tiles: {max_tiles}")
print(f"Prompt file: {prompt_file}")
print(f"Stage 1: key={stage1_key}, max_tokens={stage1_max_tokens}")
print(f"Stage 2: key={stage2_key}, max_tokens={stage2_max_tokens}")
print(f"Data mode: {data_mode}, dir: {data_dir}")
print(f"Ground truth entries: {len(ground_truth)}")

In [ ]:
"""Cell 3: Load/generate multi-receipt image."""

from experiments.synthetic.generate_multi_receipt_page import (
    generate_multi_receipt_page,
)

if data_mode == "real":
    receipt_path = data_dir / exp_config["data"]["real_receipt"]
    if not receipt_path.exists():
        raise FileNotFoundError(f"Real receipt not found: {receipt_path}")
    print(f"Using real receipt: {receipt_path}")
else:
    mr_filename = (
        staged.get("synthetic", {})
        .get("receipt", {})
        .get("filename", "synthetic_multi_receipt_page.png")
    )
    receipt_path = generate_multi_receipt_page(data_dir / mr_filename)
    print(f"Generated multi-receipt page: {receipt_path}")

receipt_img = Image.open(receipt_path)
display(receipt_img)

In [ ]:
"""Cell 4: Load/generate bank statement image."""

from experiments.synthetic.generate_bank_statement import generate_bank_statement

if data_mode == "real":
    statement_path = data_dir / exp_config["data"]["real_statement"]
    if not statement_path.exists():
        raise FileNotFoundError(f"Real bank statement not found: {statement_path}")
    print(f"Using real bank statement: {statement_path}")
else:
    statement_path = generate_bank_statement(data_dir / "synthetic_bank_statement.png")
    print(f"Generated synthetic bank statement: {statement_path}")

statement_img = Image.open(statement_path)
display(statement_img)

In [ ]:
"""Cell 5: Load InternVL3 model via registry."""

run_config_path = PROJECT_ROOT / "config" / "run_config.yml"
with run_config_path.open() as f:
    run_config = yaml.safe_load(f)

# Resolve model path
model_path = run_config.get("model", {}).get("path")
if not model_path:
    model_path = (
        run_config.get("model_loading", {}).get("default_paths", {}).get(model_type)
    )
if not model_path:
    raise FileNotFoundError(
        f"No model path found for '{model_type}' in run_config.yml. "
        "Set model.path or model_loading.default_paths.{model_type}."
    )
model_path = Path(model_path)
print(f"Model path: {model_path}")

cfg = PipelineConfig(
    data_dir=data_dir,
    output_dir=data_dir,
    model_path=model_path,
    model_type=model_type,
    max_tiles=max_tiles,
    flash_attn=exp_config["model"]["flash_attn"],
    dtype=exp_config["model"]["dtype"],
    max_new_tokens=max_new_tokens,
)

registration = get_model(model_type)
model_ctx = registration.loader(cfg)
model, tokenizer = model_ctx.__enter__()
print(f"Model loaded: {type(model).__name__}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
"""Cell 6: Stage 1 — Preprocess receipt image ONLY (no bank statement)."""

preprocessor = InternVL3ImagePreprocessor(max_tiles=max_tiles)

receipt_pv = preprocessor.load_image(str(receipt_path), model, max_num=max_tiles)
receipt_patches = [receipt_pv.shape[0]]

print(f"Receipt tiles: {receipt_pv.shape[0]}")
print(f"Receipt pv shape: {receipt_pv.shape}")
print("(Bank statement NOT loaded yet — Stage 1 uses receipt only)")

In [ ]:
"""Cell 7: Stage 1 — Load prompt and run inference on receipt."""

stage1_prompt = SimplePromptLoader.load_prompt(prompt_file, stage1_key)
stage1_question = f"<image>\n{stage1_prompt}"

print(f"Stage 1 prompt ({len(stage1_prompt)} chars):")
print("=" * 60)
print(stage1_prompt)
print("=" * 60)

stage1_gen_config = {
    "max_new_tokens": stage1_max_tokens,
    "do_sample": False,
    "num_beams": 1,
}

print("\nRunning Stage 1 inference (receipt only)...")
stage1_response = model.chat(
    tokenizer,
    receipt_pv,
    stage1_question,
    stage1_gen_config,
    num_patches_list=receipt_patches,
)
print("Stage 1 inference complete.")

In [ ]:
"""Cell 8: Stage 1 — Display and parse results (STORE/DATE/TOTAL)."""


def parse_stage1_response(text: str) -> list[dict[str, str]]:
    """Parse Stage 1 response into list of {STORE, DATE, TOTAL} dicts.

    Handles both delimited and markdown header formats:
      --- RECEIPT 1 ---
      #### Receipt 1
    """
    sections = re.split(r"---\s*RECEIPT\s+\d+\s*---", text)
    if len(sections) <= 1:
        sections = re.split(r"#{2,4}\s+Receipt\s+\d+", text)
    results = []
    for section in sections[1:]:
        entry: dict[str, str] = {}
        for line in section.splitlines():
            # Markdown bold: - **STORE:** value  or  **STORE:** value
            md_match = re.match(r"^[-\s]*\*\*([A-Z_]+):\*\*\s*(.+)$", line.strip())
            if md_match:
                entry[md_match.group(1)] = md_match.group(2).strip()
                continue
            # Plain: STORE: value
            plain_match = re.match(r"^([A-Z_]+):\s*(.+)$", line.strip())
            if plain_match:
                entry[plain_match.group(1)] = plain_match.group(2).strip()
        if entry:
            results.append(entry)
    return results


print("Stage 1 raw response:")
print("=" * 60)
print(stage1_response)
print("=" * 60)

stage1_receipts = parse_stage1_response(stage1_response)
print(f"\nParsed {len(stage1_receipts)} receipts:")
for i, r in enumerate(stage1_receipts, 1):
    print(
        f"  Receipt {i}: {r.get('STORE', '?')} | {r.get('DATE', '?')} | {r.get('TOTAL', '?')}"
    )

In [ ]:
"""Cell 9: Stage 2 — Build dynamic prompt with Stage 1 results injected."""

# Format Stage 1 results as text for injection into Stage 2 template
purchases_lines = []
for i, r in enumerate(stage1_receipts, 1):
    purchases_lines.append(
        f"Purchase {i}: Store={r.get('STORE', 'UNKNOWN')}, "
        f"Date={r.get('DATE', 'UNKNOWN')}, "
        f"Total={r.get('TOTAL', 'UNKNOWN')}"
    )
purchases_text = "\n".join(purchases_lines)

print("Purchases text for Stage 2:")
print(purchases_text)

# Load Stage 2 template and inject purchases
stage2_template = SimplePromptLoader.load_prompt(prompt_file, stage2_key)
stage2_prompt = stage2_template.format(purchases_text=purchases_text)

print(f"\nStage 2 prompt ({len(stage2_prompt)} chars):")
print("=" * 60)
print(stage2_prompt)
print("=" * 60)

In [ ]:
"""Cell 10: Stage 2 — Preprocess bank statement image ONLY."""

statement_pv = preprocessor.load_image(str(statement_path), model, max_num=max_tiles)
statement_patches = [statement_pv.shape[0]]

print(f"Statement tiles: {statement_pv.shape[0]}")
print(f"Statement pv shape: {statement_pv.shape}")
print("(Receipt pixel_values NOT included — Stage 2 uses statement only)")

In [ ]:
"""Cell 11: Stage 2 — Run inference on bank statement with dynamic prompt."""

stage2_question = f"<image>\n{stage2_prompt}"

stage2_gen_config = {
    "max_new_tokens": stage2_max_tokens,
    "do_sample": False,
    "num_beams": 1,
}

print("Running Stage 2 inference (bank statement only)...")
stage2_response = model.chat(
    tokenizer,
    statement_pv,
    stage2_question,
    stage2_gen_config,
    num_patches_list=statement_patches,
)
print("Stage 2 inference complete.")

In [ ]:
"""Cell 12: Stage 2 — Display and parse matching results."""


def parse_kv_response(text: str) -> dict[str, str]:
    """Extract KEY: VALUE pairs from model response."""
    results: dict[str, str] = {}
    for line in text.splitlines():
        md_match = re.match(r"^[-\s]*\*\*([A-Z_]+):\*\*\s*(.+)$", line.strip())
        if md_match:
            results[md_match.group(1)] = md_match.group(2).strip()
            continue
        plain_match = re.match(r"^([A-Z_]+):\s*(.+)$", line.strip())
        if plain_match:
            results[plain_match.group(1)] = plain_match.group(2).strip()
    return results


def parse_multi_receipt_response(text: str) -> list[dict[str, str]]:
    """Split response on receipt headers and parse each section."""
    sections = re.split(r"---\s*RECEIPT\s+\d+\s*---", text)
    if len(sections) <= 1:
        sections = re.split(r"#{2,4}\s+Receipt\s+\d+", text)
    results = []
    for section in sections[1:]:
        parsed = parse_kv_response(section)
        if parsed:
            results.append(parsed)
    return results


print("Stage 2 raw response:")
print("=" * 60)
print(stage2_response)
print("=" * 60)

parsed_receipts = parse_multi_receipt_response(stage2_response)
print(f"\nParsed {len(parsed_receipts)} receipt matches:")
for i, p in enumerate(parsed_receipts, 1):
    print(f"\n--- Receipt {i} ---")
    for key, value in p.items():
        print(f"  {key}: {value}")

In [ ]:
"""Cell 13: Validate against ground truth."""

print("Validation Results")
print("=" * 60)
all_pass = True

gt_list = ground_truth
gt_by_total = {gt["receipt_total"]: gt for gt in gt_list}

matched_count = 0
for i, parsed_r in enumerate(parsed_receipts, 1):
    total_key = parsed_r.get("RECEIPT_TOTAL", "")
    gt_match = gt_by_total.get(total_key)
    print(f"\n--- Receipt {i} (total: {total_key}) ---")
    if gt_match is None:
        print(f"  [WARN] No ground truth entry for total '{total_key}'")
        all_pass = False
        continue
    matched_count += 1
    gt_norm = {k.upper(): v for k, v in gt_match.items()}
    for field, expected in gt_norm.items():
        actual = parsed_r.get(field, "MISSING")
        match = actual.upper() == expected.upper()
        status = "PASS" if match else "FAIL"
        if not match:
            all_pass = False
        print(f"  [{status}] {field}: expected='{expected}' got='{actual}'")

if matched_count < len(gt_list):
    all_pass = False
    print(f"\n  [FAIL] Only matched {matched_count}/{len(gt_list)} receipts")

print("=" * 60)
if all_pass:
    print("ALL FIELDS MATCH — experiment passed.")
else:
    print("SOME FIELDS DID NOT MATCH — review above.")

In [ ]:
"""Cell 14: Comparison summary — single-pass vs staged approach."""

# Tile counts
stage1_tiles = receipt_pv.shape[0]
stage2_tiles = statement_pv.shape[0]
staged_total_tiles = stage1_tiles + stage2_tiles

print("Approach Comparison: Single-Pass vs Staged")
print("=" * 60)
print()
print("                    Single-Pass    Staged")
print("  Inference calls:  1              2")
print(f"  Stage 1 tiles:    —              {stage1_tiles} (receipt only)")
print(f"  Stage 2 tiles:    —              {stage2_tiles} (statement only)")
print(f"  Max tiles/call:   {staged_total_tiles:<14} {max(stage1_tiles, stage2_tiles)}")
print(f"  Total tiles:      {staged_total_tiles:<14} {staged_total_tiles} (same data)")
print()
print("Key differences:")
print(
    f"  - Single-pass: {staged_total_tiles} tiles in ONE call (shared resolution budget)"
)
print(
    f"  - Staged: max {max(stage1_tiles, stage2_tiles)} tiles per call (full resolution per image)"
)
print("  - Staged Stage 1 is inspectable before Stage 2 runs")
print("  - Staged Stage 2 uses TEXT from receipts, not pixels")
print()
print(f"Stage 1 receipts found: {len(stage1_receipts)}")
print(f"Stage 2 matches found:  {len(parsed_receipts)}")
print(f"Ground truth entries:   {len(ground_truth)}")
print(f"Validation:             {'PASS' if all_pass else 'FAIL'}")

In [ ]:
"""Cell 15: GPU cleanup."""

import gc

try:
    model_ctx.__exit__(None, None, None)
except Exception:
    pass

del receipt_pv, statement_pv
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Cleanup complete.")